# OGC Application Package Examples

This tutorial walks through a runnable example OGC application package. If you are new to OGC on the MAAP, this is a good place to start so you can see what an OGC application package looks like before creating your own.

All of the examples are available in the [ogc-app-pack-examples](https://github.com/marjo-luc/ogc-app-pack-examples) repository. Each example is small and self-contained, and each is bundled with everything needed to register and run it as an OGC process on MAAP.

> **New to OGC on MAAP?** See [OGC on the MAAP](../ogc.rst) for the big picture, and [Register Algorithm as an OGC Application Package](./build_application_packages.ipynb) and the [OGC-App-Pack GitHub Action](./ogc_app_pack_gha.ipynb) tutorials for the two ways to build one.

The repository contains four examples:

| Example | What it does | Runtime | Highlights |
|---------|--------------|---------|------------|
| Write String to File | Writes an input string to a text file. | Python script | Simplest example: string in, file out |
| Estimate Pi | Estimates π by numerically integrating 4/(1+x²) over [0, 1] (midpoint rule). | Compiled Fortran | A non-Python, compiled-binary process |
| Color to Greyscale | Converts an input image to greyscale using GDAL. | Python notebook (papermill) | `File` input; Jupyter notebook |
| STAC Raster Subset | Clips a raster asset from a staged STAC Catalog to a bbox and emits a new STAC Catalog. | Python notebook (papermill) | STAC I/O; Jupyter notebook |

A key takeaway from this table: **an OGC application package does not care what language or tool your algorithm is written in.** Whether your algorithm is a Python script, a compiled Fortran binary, or a Jupyter notebook, the packaging pattern is the same. The container holds your software environment, and a small entrypoint wires MAAP's inputs to your code.

The rest of this tutorial walks through the **STAC Raster Subset** (`stac_clip`) example in detail, because it exercises the most moving parts: a parameterized notebook, a container, STAC data input and output, and the MAAP stage-in / stage-out convention.

## Walkthrough: the `stac_clip` example

### What the algorithm does

The `stac_clip` algorithm performs a common geospatial operation: it clips (subsets) a raster to a bounding box. The algorithm:

1. Reads an input [STAC](https://stacspec.org/) Catalog.
2. Takes the first STAC Item in that catalog and finds the requested raster asset.
3. Reads only the window of pixels that fall inside the requested bounding box and writes them out as a Cloud-Optimized GeoTIFF (COG).
4. Emits a new STAC Catalog describing the clipped output.

### Anatomy

The `stac_clip` example is made up of a handful of files:

| File | Role |
|------|------|
| `stac_clip.ipynb` | The **algorithm** itself — a Jupyter notebook that does the clipping. |
| `run.py` | The **entrypoint** — maps command-line inputs to notebook parameters via [papermill](https://papermill.readthedocs.io/). |
| `requirements.txt` | The Python **dependencies** the notebook needs. |
| `Containerfile` | Recipe for the **container image** that bundles the code and its dependencies. |
| `algorithm_config.yml` | The **algorithm description** — inputs, outputs, resource requirements, metadata. This is required to register the algorithm to the MAAP. |
| `cwl_workflows/process_stac-clip_main.cwl` | The **CWL workflow** — *generated* from the config; you do not write this by hand. |

The sections below walk through each of these in turn, following the flow of a job: from the notebook that does the work, out to the CWL workflow that MAAP executes.

### 1. The algorithm: `stac_clip.ipynb`

The algorithm is an ordinary Jupyter notebook. It contains a parameters cell, tagged `parameters`, at the top. Papermill uses this tag to know which cell to overwrite with values supplied at run time. The values you see here are just defaults for interactive development — they are replaced when the algorithm runs.

```python
# Papermill parameters cell -- values here are overridden at execution time.
input_catalog = "input"  # path to the STAC Catalog directory
asset_name = "B04"
bbox = "-122.55 37.70 -122.35 37.85"  # MINX MINY MAXX MAXY, EPSG:4326
output_file = "clipped.tif"
```

The rest of the notebook is plain Python organized into three steps. First, it **reads the staged catalog** and resolves the requested asset's on-disk path. Because MAAP stages the input data into the container before the notebook runs (more on that below), the asset's href points to a local file:

```python
def read_input_item(input_catalog, asset_name):
    """Load the staged catalog and return (item, local asset href) for ``asset_name``."""
    catalog_path = input_catalog
    if os.path.isdir(catalog_path):
        catalog_path = os.path.join(catalog_path, "catalog.json")
    catalog = pystac.Catalog.from_file(catalog_path)
    items = list(catalog.get_items(recursive=True))
    if not items:
        raise LookupError(f"No STAC Items found in catalog {catalog_path!r}.")
    item = items[0]
    if asset_name not in item.assets:
        raise KeyError(
            f"Asset {asset_name!r} not found. Available: {sorted(item.assets)}"
        )
    href = item.assets[asset_name].get_absolute_href() or item.assets[asset_name].href
    print(f"Input item {item.id!r}; asset {asset_name!r} -> {href}")
    return item, href
```

Next, it **clips the raster**:

```python
def clip_asset(href, bbox, output_path):
    """Window-read ``href`` to ``bbox`` (lon/lat) and write it as a COG."""
    with rasterio.open(href) as src:
        # The bbox is in EPSG:4326; reproject it to the raster's CRS.
        dst_bounds = transform_bounds("EPSG:4326", src.crs, *bbox)
        window = from_bounds(*dst_bounds, transform=src.transform)
        window = window.round_offsets().round_lengths()

        if window.width <= 0 or window.height <= 0:
            raise ValueError(
                f"bbox {bbox} does not intersect the raster bounds {src.bounds}."
            )

        data = src.read(window=window)
        transform = src.window_transform(window)
        ...
```

Finally, it **writes a new STAC Catalog** (the "stage-out" side) describing the clipped COG. The whole notebook is driven by a short run cell at the bottom that parses the `bbox` string and calls the three functions in order.

### 2. The entrypoint: `run.py`

The MAAP runs your algorithm by invoking a single command and passing each input as a named command-line argument (for example, `--bbox "-122.55 37.70 -122.35 37.85"`). A notebook cannot be invoked that way directly, so `run.py` bridges the two: it parses the command-line arguments and hands them to papermill, which injects them into the notebook's `parameters` cell and executes it.

```python
#!/usr/bin/env python3
"""Entrypoint that maps the OGC ``--<name> value`` inputs onto papermill
parameters and executes the stac_clip notebook."""

import argparse
import os

import papermill as pm

NOTEBOOK_PATH = "/app/stac_clip.ipynb"
OUTPUT_DIR = "output"


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Execute the stac_clip notebook with papermill."
    )
    parser.add_argument("--input_catalog", required=True,
                        help="Path to the STAC Catalog directory.")
    parser.add_argument("--asset_name",
                        help="Name of the raster asset to clip (default: B04).")
    parser.add_argument("--bbox", required=True,
                        help="Clip bounding box as 'MINX MINY MAXX MAXY' in EPSG:4326.")
    parser.add_argument("--output_file", default="clipped.tif",
                        help="Name of the output COG (default: clipped.tif).")
    args = parser.parse_args()

    # The notebook writes results into ./output; keep the executed copy there too.
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    executed_notebook = os.path.join(OUTPUT_DIR, "executed_stac_clip.ipynb")

    pm.execute_notebook(
        NOTEBOOK_PATH,
        executed_notebook,
        parameters={
            "input_catalog": args.input_catalog,
            "asset_name": args.asset_name,
            "bbox": args.bbox,
            "output_file": args.output_file,
        },
    )


if __name__ == "__main__":
    main()
```

The results are written into an `output/` directory. That directory is what MAAP collects when the job finishes.

> If your algorithm is a plain script or a compiled binary rather than a notebook, the entrypoint is even simpler — it just needs to accept named arguments. Compare with the `write_string_to_file` and `estimate_pi` examples, whose entrypoints run a python script and a Fortran binary respectively.

### 3. The container: `Containerfile` and `requirements.txt`

The container image is the reproducible, portable environment your algorithm runs in. It pins your operating-system packages, your language runtime, and your dependencies, so the algorithm behaves the same way no matter where it runs.

The `requirements.txt` lists the Python packages the notebook needs:

```
rasterio
rio-stac
pystac
papermill
ipykernel
```

The `Containerfile` (equivalent to a `Dockerfile`) is the recipe for building the image. It starts from a slim Python base, installs a system library that `rasterio`'s bundled GDAL needs, installs the Python dependencies, registers a Jupyter kernel so papermill can execute the notebook, and finally copies in the notebook and its entrypoint:

```dockerfile
FROM python:3.12-slim

# python:3.12-slim omits libexpat1, so install it here for rasterio's bundled GDAL
RUN apt-get update \
    && apt-get install -y --no-install-recommends libexpat1 \
    && rm -rf /var/lib/apt/lists/*

COPY ./stac_clip/requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r /app/requirements.txt \
    && python3 -m ipykernel install --name python3

# Notebook and its entrypoint wrapper
COPY ./stac_clip/stac_clip.ipynb /app/stac_clip.ipynb
COPY ./stac_clip/run.py /usr/local/bin/run.py

RUN chmod +x /usr/local/bin/run.py
```

You do not need to build this image yourself — the build step is handled automatically when you deploy (see the next section). But it is worth understanding what it contains, because **the container is what makes your algorithm reproducible and portable.** Anyone with the image can run your algorithm and get the same result.

### 4. The algorithm description: `algorithm_config.yml`

This YAML file is the description of your algorithm -- its inputs and outputs, and the resources it needs. MAAP reads this file to generate the CWL workflow and to build the registration.

```yaml
algorithm_description: Reads a STAC Catalog, clips the requested raster asset to a bounding box, and emits a new STAC Catalog describing the clipped output.
algorithm_name: stac-clip
algorithm_version: main
keywords: ogc, stac, raster
code_repository: https://github.com/marjo-luc/ogc-app-pack-examples.git
citation: https://github.com/marjo-luc/ogc-app-pack-examples.git
author: mlucas
contributor: mlucas
license: https://raw.githubusercontent.com/marjo-luc/ogc-app-pack-examples/refs/heads/main/LICENSE
release_notes: None
run_command: run.py
ram_min: 5      # mebibytes
cores_min: 1
outdir_max: 10  # mebibytes

inputs:
  - name: input_catalog
    doc: Path to the STAC Catalog directory
    label: Input STAC Catalog
    type: Directory
  - name: asset_name
    doc: Name of the raster asset to clip
    label: Asset name
    type: string
  - name: bbox
    doc: Clip bounding box as 'MINX MINY MAXX MAXY' in EPSG:4326
    label: Bounding box
    type: string
  - name: output_file
    doc: Name of the output COG
    label: Output filename
    type: string?
    default: clipped.tif

outputs:
  - name: out
    type: Directory
```

The fields fall into three groups:

- **Identity & metadata** — `algorithm_name`, `algorithm_version`, `algorithm_description`, `code_repository`, `author`, `license`, etc.. This is what shows up when someone discovers your algorithm in the catalog.
- **Execution** — `run_command` is the command MAAP invokes (here, `run.py`); `ram_min`, `cores_min`, and `outdir_max` declare the resources the job needs.
- **Inputs & outputs** — each input has a `name` (which must match the argument in `run.py`), a human-readable `label`, a `doc` string, and a `type`. Note the types:
  - `input_catalog` is a `Directory` — this signals to MAAP that the input is a dataset that must be *staged in* (downloaded) before the job runs.
  - `asset_name` and `bbox` are plain `string` values.
  - `output_file` is a `string?` — the trailing `?` marks it optional, and `default: clipped.tif` supplies a value when the caller omits it.
  - The single output `out` is a `Directory` — DPS collects everything the job writes to that directory and *stages it out* to your workspace.

### 5. From config to a deployed process: the CWL workflow

You have the notebook, the entrypoint, the container recipe, and the algorithm description. The last step is to turn all of that into a deployed OGC process on MAAP. This is where the [OGC App Pack Generator GitHub Action](./ogc_app_pack_gha.ipynb) comes in.

In the examples repository, pushing a change under an example directory triggers a per-example GitHub workflow that runs the generator action. That action:

1. Reads `algorithm_config.yml` and the `Containerfile`.
2. Builds and publishes the container image (for `stac_clip`, to `ghcr.io/marjo-luc/stac-clip:main`).
3. Generates the CWL workflow from the config and commits it back to the repo under `cwl_workflows/`.
4. Registers the process with the MAAP OGC processes endpoint so it can be run as a job.

You do not write the CWL by hand — but it helps to see what is generated, because the CWL is the platform-neutral contract that any OGC-compliant runner (including MAAP's DPS) uses to execute your algorithm. Here is the workflow generated for `stac_clip`:

```yaml
cwlVersion: v1.2
$graph:
- class: Workflow
  label: stac-clip
  doc: Reads a STAC Catalog, clips the requested raster asset to a bounding box, and
    emits a new STAC Catalog describing the clipped output.
  id: stac-clip
  inputs:
    input_catalog: { doc: Path to the STAC Catalog directory, label: Input STAC Catalog, type: Directory }
    asset_name:    { doc: Name of the raster asset to clip, label: Asset name, type: string }
    bbox:          { doc: 'Clip bounding box as ...', label: Bounding box, type: string }
    output_file:   { doc: Name of the output COG, label: Output filename, type: string?, default: clipped.tif }
  outputs:
    out: { type: Directory, outputSource: process/outputs_result }
  steps:
    process:
      run: '#main'
      in: { input_catalog: input_catalog, asset_name: asset_name, bbox: bbox, output_file: output_file }
      out: [outputs_result]
- class: CommandLineTool
  id: main
  requirements:
    DockerRequirement:
      dockerPull: ghcr.io/marjo-luc/stac-clip:main
    NetworkAccess: { networkAccess: true }
    ResourceRequirement: { ramMin: 5, coresMin: 1, outdirMax: 10 }
  baseCommand: run.py
  inputs:
    input_catalog: { type: Directory, inputBinding: { position: 1, prefix: --input_catalog } }
    asset_name:    { type: string, inputBinding: { position: 2, prefix: --asset_name } }
    bbox:          { type: string, inputBinding: { position: 3, prefix: --bbox } }
    output_file:   { type: string?, inputBinding: { position: 4, prefix: --output_file }, default: clipped.tif }
  outputs:
    outputs_result: { outputBinding: { glob: ./output* }, type: Directory }
```

Reading this top to bottom, you can now see the whole package connect up:

- The `Workflow` block mirrors the `inputs`/`outputs` from your `algorithm_config.yml`.
- The `CommandLineTool` block ties it to execution: `dockerPull` is the image that was built from your `Containerfile`; `baseCommand: run.py` is your `run_command`; and each `inputBinding` turns an input into the `--<name> value` argument that `run.py` parses.
- The output `glob: ./output*` is how the runner collects the `output/` directory your notebook wrote — the stage-out step.

> **The two ways to build.** This example uses the GitHub Action for a fully automated, push-to-deploy workflow. You can also build an application package interactively from inside a MAAP workspace using the Algorithm Catalog plugin — see [Register Algorithm as an OGC Application Package](./build_application_packages.ipynb). Both paths consume the same `algorithm_config.yml` and produce the same kind of deployed process.

### 6. Stage-in / stage-out, and running the job

Your algorithm does not need to download its own input data or upload its own results. Instead:

- **Stage-in:** before your container starts, the MAAP localizes each `Directory` input onto the container's filesystem. For `stac_clip`, MAAP takes the input STAC Item and writes a local catalog directory, so that by the time the notebook runs, `input_catalog` points at real files on disk. This is why `read_input_item` can simply open the asset href as a local file.
- **Stage-out:** when your container exits, MAAP collects everything written to the output `Directory` (matched by `glob: ./output*`) and copies it to your workspace, under `~/my-private-bucket/dps_output`, organized by algorithm name and job tag.

This separation is what keeps an application package portable: your code just reads local files and writes local files, and the platform handles moving data in and out.

**Running the algorithm as a job on MAAP.** Once the process is registered, you run it like any other MAAP algorithm — open the **Submit Jobs** plugin, pick the `stac-clip` algorithm and version, choose a resource queue, and fill in the inputs (`input_catalog`, `asset_name`, `bbox`, and optionally `output_file`). See [Submit a Job](./submit_job.ipynb) for the full walkthrough, including how to find your output files afterward.

**Running the CWL locally.** Because the generated CWL is platform-neutral, you can also run it on your own machine with the reference runner, [cwltool](https://github.com/common-workflow-language/cwltool), plus Docker or Podman. Each example ships a sample job file, `input.yml`:

```yaml
input_catalog:
  class: Directory
  location: https://cmr.earthdata.nasa.gov/stac/LPCLOUD/collections/HLSL30_2.0/items/HLS.L30.T10SEG.2023198T184546.v2.0
asset_name: B04
bbox: -122.55 37.70 -122.35 37.85
output_file: clipped.tif
```

Then, from the repository root:

```bash
pip install cwltool
cwltool stac_clip/cwl_workflows/process_stac-clip_main.cwl stac_clip/input.yml
```

> **Note:** `stac_clip` relies on the stage-in convention, so to run it locally, you need to emulate what MAAP does on the platform — create a directory named `input` and download the STAC item into it. The `write_string_to_file`, `estimate_pi`, and `color_to_greyscale` examples run locally with no extra setup.

## Next steps

You have now seen a complete application package end to end: a notebook that does the work, an entrypoint that wires MAAP's inputs to it, a container that makes it reproducible, a config that describes it, and the generated CWL that MAAP executes.

To go further:

- **Try the examples yourself** — clone [ogc-app-pack-examples](https://github.com/marjo-luc/ogc-app-pack-examples) and run one locally with `cwltool`.
- **Build your own** — start from [Register Algorithm as an OGC Application Package](./build_application_packages.ipynb) (interactive) or the [OGC-App-Pack GitHub Action](./ogc_app_pack_gha.ipynb) (automated).
- **Convert an existing MAAP algorithm** — see [Transition to OGC](./transition_to_ogc.ipynb).
- **Run a job** — see [Submit a Job](./submit_job.ipynb).